## 02 - Preprocessing

**Imports and paths**

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
RAW_PATH = DATA_DIR / "Patient_Health_Records_Raw.csv"
CLEAN_PATH = DATA_DIR / "processed" / "Cleaned_Health_Records.csv"

# Placeholder tokens that mean "value not recorded"
empty = ["", "nan", "none", "null", "n/a", "na", "unknown"]

**Load data**

In [3]:
df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head()

(10000, 17)


,Patient_ID,Name,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Has_Disease
0,ID-00001,"SMITH, ALICE",250,Unknown,Boston,22kg/m2,120 over 80,83,190,Unknown,yes,NaN,Apr 15 2021,30,A00,Patient feeling fine 🙂,0
1,ID-00002,"SMITH, ALICE",-5,FEMALE,newyork,25.8,NaN,500,normal,N,No,NaN,2021-07-10,NaN,unknown,"BP high, needs recheck",unknown
2,ID-00003,John Doe,-5,NaN,Nwe Yrok,22kg/m2,120 - 80,500,190,no,Former,"aspirin,metformin",Apr 06 2021,30,B20,NaN,unknown
3,ID-00004,michael brown,80,F,New York,33.4,NaN,eighty,250,y,EX-smoker,NaN,20241113,30,NaN,Follow up soon,unknown
4,ID-00005,Jane Doe,250,male,la,30.1,120 over 80,0,normal,no,No,NaN,2025-09-27,NaN,I10,Patient feeling fine 🙂,1


**2.1 Has_Disease**

In [4]:
# Only confirmed 0/1 labels can be used for supervised training, so unknown and blank labels are dropped
def clean_target(x):
    x = str(x).strip().lower()
    if x in ['1', 'yes', 'y']:
        return 1
    elif x in ['0', 'no', 'n']:
        return 0
    else:
        return np.nan

df['Has_Disease'] = df['Has_Disease'].apply(clean_target)
# Keep the original row index so Patient_ID gaps can be rebuilt from row position (2.16)
df = df.dropna(subset=['Has_Disease'])
df['Has_Disease'] = df['Has_Disease'].astype(int)
print(df.shape)
df['Has_Disease'].value_counts()

(4974, 17)


Has_Disease
0    2513
1    2461
Name: count, dtype: int64

**2.2 Age**

In [5]:
# Ages are recorded as words, x10 entry errors (250) and negative sentinels (-5)
age_words = {'twenty': 20, 'thirty': 30, 'forty': 40, 'fourty': 40, 'fifty': 50}

def clean_age(x):
    x = str(x).strip().lower()
    if x in age_words:
        return age_words[x]
    try:
        val = float(x)
    except ValueError:
        return np.nan
    if val < 0:
        return np.nan
    return val / 10 if val > 120 else val

df['Age'] = df['Age'].apply(clean_age)
df['Age'] = df['Age'].fillna(df['Age'].mode()[0])

**2.3 Gender**

In [6]:
# Collapse case and abbreviation variants into M/F, then fill unknowns with the most common gender
gender ={
    'male':'M',
    'female':'F',
    'm':'M',
    'f':'F',
}
df['Gender']=df['Gender'].astype(str).str.lower().str.strip()
df['Gender']=df['Gender'].replace(gender)
df['Gender']=df['Gender'].replace(['unknown','nan','Nan',''],np.nan)
df['Gender']=df['Gender'].fillna(df['Gender'].mode()[0])

**2.4 City**

In [7]:
# Merge abbreviations and typos (NY, newyork, Nwe Yrok) into one city name
def city_standard(x):
  x=str(x).lower()
  x=str(x).replace(' ','')
  x=str(x).strip()
  if x in empty:
    return np.nan
  city ={
      'boston':'Boston',
      'la'  :'Los Angeles',
      'losangeles':'Los Angeles',
      'newyork':'New York',
      'nweyrok':'New York',
      'ny':'New York',
      'chicago':'Chicago',
  }
  return city.get(x,x.title())
df['City']=df['City'].apply(city_standard)
df['City']=df['City'].fillna(df['City'].mode()[0])

**2.5 BMI**

In [8]:
# Strip the kg/m2 unit so every BMI is a plain number
def BMI_standard(x):
  x=str(x).lower()
  x=str(x).strip()
  if x in empty:
    return np.nan
  elif 'kg/m2' in x:
    x=x.replace('kg/m2','')
  return float(x)

df['BMI']=df['BMI'].apply(BMI_standard)
df['BMI']=df['BMI'].fillna(df['BMI'].mean())

**2.6 Blood_Pressure**

In [9]:
# Unify '120 over 80' and '120 - 80' into the standard 'systolic/diastolic' format
def correct_bp(x):
  x=str(x).lower()
  x=str(x).strip()
  if x in empty:
    return np.nan
  elif 'over' in x:
    x=x.replace(' over ','/')
    return x
  elif '-' in x:
    x=x.replace(' - ','/')
    return x
  else:
    return x

df['Blood_Pressure']=df['Blood_Pressure'].apply(correct_bp)
df['Blood_Pressure']=df['Blood_Pressure'].fillna(df['Blood_Pressure'].mode()[0])

**2.7 Heart_Rate**

In [10]:
# Keep numeric rates in a realistic human range; sentinels (0, 500) and words become missing
def clean_heart_rate(x):
    try:
        x = str(x).strip().lower()
        if x in empty or not x.replace('.', '').isdigit():
            return np.nan
        val = float(x)
        # keep realistic human range
        if val < 40 or val > 180:
            return np.nan
        return val
    except:
        return np.nan

df['Heart_Rate'] = df['Heart_Rate'].apply(clean_heart_rate)
df['Heart_Rate'] = df['Heart_Rate'].fillna(df['Heart_Rate'].median())

**2.8 Cholesterol_Level**

In [11]:
# Convert text levels (high/normal/low) to representative mg/dL values
def correct_cholesterol(x):
  x=str(x).lower()
  x=str(x).strip()
  if x in empty:
    return np.nan
  cholesterol = {
    'high':'250',
    'low' : '100',
    'normal':'125'
}

  return cholesterol.get(x,x)

df['Cholesterol_Level']=df['Cholesterol_Level'].apply(correct_cholesterol)
df['Cholesterol_Level']=df['Cholesterol_Level'].fillna(df['Cholesterol_Level'].mode()[0])
df['Cholesterol_Level']=df['Cholesterol_Level'].astype(float)

**2.9 Diabetic**

In [12]:
# Encode yes/y and no/n variants as binary 1/0
def clean_diabetic(x):
    x = str(x).strip().lower()
    if x in empty:
        return np.nan
    mapping = {'yes': "Y", 'y': "Y", 'no': "N", 'n': "N"}
    return mapping.get(x, np.nan)

df['Diabetic'] = df['Diabetic'].apply(clean_diabetic)
df['Diabetic'] = df['Diabetic'].fillna(df['Diabetic'].mode()[0])

**2.10 Smoker**

In [13]:
# Encode smoking status: 0 = never, 1 = current, 2 = former (EX-smoker/Former)
def clean_smoker(x):
    x = str(x).strip().lower()
    if x in empty:
        return np.nan
    mapping = {
        'yes': "Y", 'y': "Y", 'no': "N", 'n': "N",
        'ex-smoker': "Ex", 'former': "Ex"
    }
    return mapping.get(x, np.nan)

df['Smoker'] = df['Smoker'].apply(clean_smoker)
df['Smoker'] = df['Smoker'].fillna(df['Smoker'].mode()[0])

**2.11 Medications**

In [14]:
# Unify list separators (',', ';', '/') so the same drug combination has one spelling; missing means no medication
def clean_meds(x):
    x = str(x).lower().strip()
    if x in empty:
        return np.nan
    x = x.replace(';', ',')
    x = x.replace('/', ',')
    x = x.replace(' ,', ',').replace(', ', ',')
    return x

df['Medications'] = df['Medications'].apply(clean_meds)
df['Medications'] = df['Medications'].fillna('none')

**2.12 Last_Visit_Date**

In [15]:
# Dates come in four formats; parse each value individually (slash dates are dd/mm/yyyy)
df['Last_Visit_Date'] = pd.to_datetime(df['Last_Visit_Date'], errors='coerce', format='mixed', dayfirst=True)
df['Last_Visit_Date'] = df['Last_Visit_Date'].fillna(df['Last_Visit_Date'].mode()[0])

**2.13 Follow_Up**

In [16]:
# Keep follow-up as whole days; text such as 'two weeks' becomes missing
def clean_follow_up(x):
    x = str(x).strip().lower()
    if x in empty or x == 'n/a':
        return np.nan
    try:
        return int(float(x))
    except:
        return np.nan

df['Follow_Up'] = df['Follow_Up'].apply(clean_follow_up)
df['Follow_Up'] = df['Follow_Up'].fillna(df['Follow_Up'].mode()[0])

**2.14 Diagnosis_Code**

In [17]:
# Placeholder codes are missing; label them explicitly instead of guessing a diagnosis
df['Diagnosis_Code'] = df['Diagnosis_Code'].replace(empty, np.nan)
df['Diagnosis_Code'] = df['Diagnosis_Code'].fillna('Unknown')

**2.15 Notes**

In [18]:
# Placeholder notes are missing; label them explicitly as 'No Notes'
df['Notes'] = df['Notes'].astype(str).str.strip()
df['Notes'] = df['Notes'].replace(empty, np.nan)
df['Notes'] = df['Notes'].fillna('No Notes')

**2.16 Patient_ID**

In [19]:
# Raw IDs follow the row order (ID-00001 = row 1), so blank IDs are rebuilt from the original row position
df['Patient_ID']=df['Patient_ID'].astype(str)
df['Patient_ID']=df['Patient_ID'].str.replace('ID-','',regex=False)
df['Patient_ID']=df['Patient_ID'].str.strip()
df['Patient_ID']=pd.to_numeric(df['Patient_ID'],errors='coerce')
df['Patient_ID'] = df['Patient_ID'].fillna(pd.Series(df.index + 1, index=df.index))
df['Patient_ID']=df['Patient_ID'].astype(int)
df['Patient_ID']=df['Patient_ID'].astype(str)
df['Patient_ID']='ID-'+df['Patient_ID']

**2.17 Name**

In [20]:
# Unify case and remove the 'Last, First' comma so the same person has one spelling
df['Name']=df['Name'].str.title()
df['Name']=df['Name'].str.replace(',','')
df['Name']=df['Name'].str.strip()
name_empty=['','None','Nan','nan','unknown','n/a','N/A',np.nan]
df['Name']=df['Name'].replace(name_empty,np.nan)
df['Name']=df['Name'].fillna(df['Name'].mode()[0])

**Validate cleaned data**

In [21]:
print("Missing values remaining:", df.isna().sum().sum())
df.dtypes

Missing values remaining: 0


Patient_ID                   object
Name                         object
Age                         float64
Gender                       object
City                         object
BMI                         float64
Blood_Pressure               object
Heart_Rate                  float64
Cholesterol_Level           float64
Diabetic                     object
Smoker                       object
Medications                  object
Last_Visit_Date      datetime64[ns]
Follow_Up                   float64
Diagnosis_Code               object
Notes                        object
Has_Disease                   int64
dtype: object

In [22]:
df.head()

,Patient_ID,Name,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Has_Disease
0,ID-1,Smith Alice,25.0,F,Boston,22.000000,120/80,83.0,190.0,N,Y,none,2021-04-15,30.0,A00,Patient feeling fine 🙂,0
4,ID-5,Jane Doe,25.0,M,Los Angeles,30.100000,120/80,85.0,125.0,N,N,none,2025-09-27,14.0,I10,Patient feeling fine 🙂,1
5,ID-6,Michael Brown,20.0,F,New York,22.000000,120/80,63.0,125.0,N,Ex,none,2025-05-22,30.0,E11.9,No Notes,0
8,ID-9,John Doe,25.0,F,Boston,23.075909,120/80,85.0,250.0,N,Ex,"aspirin,metformin",2020-03-02,30.0,Unknown,No Notes,0
9,ID-10,Michael Brown,20.0,F,Chicago,21.500000,120/80,85.0,250.0,N,Ex,none,2025-03-30,30.0,A00,"BP high, needs recheck",1


**Save cleaned dataset**

In [23]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved {df.shape} to {CLEAN_PATH}")

Saved (4974, 17) to ../data/processed/Cleaned_Health_Records.csv
